In [ ]:
# Edit only attached Kaggle Input paths and dense execution batch sizes.
from pathlib import Path

BM25_WHEEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/offline-packages/bm25s-0.3.11-py3-none-any.whl"
)
LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legalir/selected-contexts")
DENSE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/bge-m3-kaggle")
RERANKER_MODEL_PATH = Path(
    "/kaggle/input/datasets/mduy2911/bge-reranker-v2-m3-kaggle"
)

CORPUS_BATCH_SIZE = 256  # May be reduced for OOM; semantics do not change.
QUERY_BATCH_SIZE = 64    # May be reduced for OOM; semantics do not change.

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
EXPECTED_SOURCE_SHA256 = "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS = 8_532
EXPECTED_CHUNKS = 199_816
EXPECTED_BM25_HOLDOUT_RECALL_AT_100 = 0.9565004887585533
EXPECTED_BM25_HOLDOUT_CE_PRECISION = 0.17888563049853376
EXPECTED_BM25_HOLDOUT_CE_RECALL = 0.8456337569240794
EXPECTED_BM25_HOLDOUT_CE_MRR = 0.711260732959998

CHUNK_SIZE = 2_000
CHUNK_OVERLAP = 200
BM25_METHOD = "lucene"
BM25_K1 = 1.5
BM25_B = 0.75
TOP_K_CHUNKS = 2_000
DOCUMENT_AGGREGATION = "sum_top_2"
CANDIDATE_DEPTH = 100
SUPPORTING_CHUNKS_PER_DOCUMENT = 2
DENSE_MAX_LENGTH = 8_192
RERANKER_MAX_SEQUENCE_LENGTH = 8_192
RERANKER_BATCH_SIZE = 128
FINAL_K = 5
SMOKE_QUERIES = 2
SANITY_ABS_TOLERANCE = 1e-9
RESULT_PATH = Path(
    "/kaggle/working/dense_cross_encoder_reranking_holdout_results.json"
)


In [ ]:
# Enforce offline execution and fail loudly for missing attached artifacts.
import os
import subprocess
import sys

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

for path, description, must_be_directory in (
    (BM25_WHEEL_PATH, "pinned local BM25 wheel", False),
    (LEGALIR_SOURCE_PATH, "LegalIR source JSON", False),
    (CORPUS_PATH, "LegalIR corpus directory", True),
    (DENSE_MODEL_PATH, "complete local BGE-M3 snapshot", True),
    (RERANKER_MODEL_PATH, "complete local BGE reranker snapshot", True),
):
    exists = path.is_dir() if must_be_directory else path.is_file()
    if not exists:
        raise FileNotFoundError(f"Attach the {description} at: {path}")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-index", "--no-deps", str(BM25_WHEEL_PATH),
    ],
    check=True,
)


In [ ]:
# Standalone aggregate-only fixed-local-holdout implementation.
import gc
import json
import re
from collections import Counter, defaultdict
from hashlib import sha256
from math import isfinite
from statistics import median
from time import perf_counter

import bm25s
import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_fixed_holdout(path: Path) -> tuple[dict, dict]:
    source_hash = sha256(path.read_bytes()).hexdigest()
    if source_hash != EXPECTED_SOURCE_SHA256:
        raise RuntimeError(
            f"LegalIR source SHA-256 mismatch: expected {EXPECTED_SOURCE_SHA256}, "
            f"got {source_hash}. Stop before reading samples."
        )
    value = read_json(path)
    if not isinstance(value, dict) or not all(isinstance(v, dict) for v in value.values()):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    samples = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(samples) != len(value):
        raise ValueError("duplicate sample IDs after string canonicalization")

    counts = {"train": 0, "dev": 0, "holdout": 0}
    holdout_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = (
            question if isinstance(question, str)
            else f"\0fallback-sample-id:{sample_id}"
        )
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        split_name = "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        counts[split_name] += 1
        if split_name == "holdout":
            holdout_ids.append(sample_id)
    if counts != EXPECTED_SPLIT_COUNTS:
        raise RuntimeError(f"fixed split counts mismatch: {counts}")

    holdout_ids.sort()
    holdout = {sample_id: samples[sample_id] for sample_id in holdout_ids}
    if len(holdout) != EXPECTED_SPLIT_COUNTS["holdout"]:
        raise RuntimeError("fixed holdout query count mismatch; stop")
    for sample in holdout.values():
        if not isinstance(sample.get("question"), str):
            raise TypeError("fixed holdout question must be a string")
        if not isinstance(sample.get("answer"), list) or not sample["answer"]:
            raise ValueError(
                "fixed holdout expected a non-empty LegalIR answer list"
            )
    return holdout, {
        "name": "fixed local holdout",
        "queries": len(holdout),
        "source_sha256": source_hash,
        "split_counts": counts,
    }


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(
        item for item in path.rglob("*")
        if item.is_file() and item.suffix.lower() == ".json"
    )
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = [str(document.get("id")) for document in documents]
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    return documents


def chunk_corpus(documents: list[dict]) -> list[dict]:
    step = CHUNK_SIZE - CHUNK_OVERLAP
    if CHUNK_SIZE <= 0 or CHUNK_OVERLAP < 0 or step <= 0:
        raise ValueError("invalid fixed-window chunk controls")
    chunks = []
    for document in documents:
        document_id = str(document["id"])
        passage = document.get("passage")
        if not isinstance(passage, str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        for chunk_index, start in enumerate(range(0, len(passage), step)):
            end = min(start + CHUNK_SIZE, len(passage))
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "text": passage[start:end],
            })
            if end == len(passage):
                break
    return chunks


def aggregate_hits(chunk_hits: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for hit in chunk_hits:
        if not isfinite(float(hit["score"])):
            raise RuntimeError("retrieval produced a non-finite score")
        grouped[hit["document_id"]].append(hit)
    documents = []
    for document_id, hits in grouped.items():
        ordered = sorted(
            hits, key=lambda hit: (-hit["score"], hit["chunk_rank"], hit["chunk_index"])
        )
        support = ordered[:SUPPORTING_CHUNKS_PER_DOCUMENT]
        documents.append({
            "document_id": document_id,
            "retrieval_score": sum(hit["score"] for hit in support),
            "best_chunk_rank": min(hit["chunk_rank"] for hit in hits),
            "supporting_chunk_indices": [hit["chunk_index"] for hit in support],
            "supporting_chunk_scores": [float(hit["score"]) for hit in support],
        })
    documents.sort(
        key=lambda document: (
            -document["retrieval_score"],
            document["best_chunk_rank"],
            document["document_id"],
        )
    )
    selected = documents[:CANDIDATE_DEPTH]
    if len(selected) != CANDIDATE_DEPTH:
        raise RuntimeError(f"expected {CANDIDATE_DEPTH} candidate documents")
    for original_rank, document in enumerate(selected, start=1):
        document["original_rank"] = original_rank
    ids = [document["document_id"] for document in selected]
    if len(ids) != len(set(ids)):
        raise RuntimeError("candidate ranking contains duplicate document IDs")
    return selected


def package_candidates(candidates_by_query: dict, seconds: float) -> dict:
    return {
        "candidates": candidates_by_query,
        "rankings": {
            sample_id: [document["document_id"] for document in documents]
            for sample_id, documents in candidates_by_query.items()
        },
        "seconds": seconds,
    }


TOKEN_PATTERN = re.compile(r"\w+", flags=re.UNICODE)


def lexical_tokenize(text: str) -> list[str]:
    return TOKEN_PATTERN.findall(text.lower())


def build_bm25(chunks: list[dict]) -> dict:
    if bm25s.__version__ != "0.3.11":
        raise RuntimeError(f"expected bm25s==0.3.11, got {bm25s.__version__}")
    started = perf_counter()
    tokenized = bm25s.tokenize(
        [chunk["text"] for chunk in chunks],
        lower=True, token_pattern=r"(?u)\w+", stopwords=[], stemmer=None,
        return_ids=True, show_progress=False,
    )
    retriever = bm25s.BM25(k1=BM25_K1, b=BM25_B, method=BM25_METHOD)
    retriever.index(tokenized, show_progress=False)
    return {"retriever": retriever, "build_seconds": perf_counter() - started}


def retrieve_bm25(index: dict, chunks: list[dict], samples: dict) -> dict:
    started = perf_counter()
    candidates_by_query = {}
    items = list(samples.items())
    for batch_start in range(0, len(items), QUERY_BATCH_SIZE):
        batch = items[batch_start:batch_start + QUERY_BATCH_SIZE]
        result = index["retriever"].retrieve(
            [lexical_tokenize(sample["question"]) for _, sample in batch],
            k=TOP_K_CHUNKS, sorted=True, return_as="tuple", show_progress=False,
        )
        for (sample_id, _), indices, scores in zip(batch, result.documents, result.scores):
            hits = []
            for rank, (index_value, score_value) in enumerate(zip(indices, scores), start=1):
                chunk_index = int(index_value)
                hits.append({
                    "document_id": chunks[chunk_index]["document_id"],
                    "chunk_index": chunk_index,
                    "score": float(score_value),
                    "chunk_rank": rank,
                })
            candidates_by_query[sample_id] = aggregate_hits(hits)
    return package_candidates(candidates_by_query, perf_counter() - started)


def model_metadata(model, model_name: str, declared_revision: str, path: Path) -> dict:
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is None:
        status = "declared-offline-snapshot"
    elif config_hash == declared_revision:
        status = "verified-from-config"
    else:
        raise RuntimeError(
            f"{model_name} config _commit_hash {config_hash!r} does not match "
            f"declared revision {declared_revision!r}"
        )
    return {
        "model_name": model_name,
        "declared_revision": declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": status,
        "local_input_path": str(path),
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(
        DENSE_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if tokenizer_limit < DENSE_MAX_LENGTH or model_limit < DENSE_MAX_LENGTH:
        raise RuntimeError("local dense model does not support max_length=8192")
    metadata = model_metadata(
        model, DENSE_MODEL_NAME, DENSE_DECLARED_REVISION, DENSE_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer, "model": model, "device": "cuda",
        "dtype": "float16", "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def load_reranker() -> dict:
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(
        RERANKER_MODEL_PATH, local_files_only=True
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    tokenizer_limit = int(tokenizer.model_max_length)
    model_limit = int(getattr(model.config, "max_position_embeddings", tokenizer_limit))
    if tokenizer_limit < RERANKER_MAX_SEQUENCE_LENGTH or model_limit < RERANKER_MAX_SEQUENCE_LENGTH:
        raise RuntimeError("local reranker does not support max_sequence_length=8192")
    metadata = model_metadata(
        model, RERANKER_MODEL_NAME, RERANKER_DECLARED_REVISION, RERANKER_MODEL_PATH
    )
    model.to("cuda")
    model.eval()
    return {
        "tokenizer": tokenizer, "model": model, "device": "cuda",
        "dtype": "float16", "metadata": metadata,
        "load_seconds": perf_counter() - started,
    }


def move_model(bundle: dict, device: str) -> None:
    bundle["model"].to(device)
    bundle["device"] = device
    bundle["model"].eval()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def encode_normalized_cls(
    dense_model: dict, texts: list[str], batch_size: int, collect_lengths: bool
) -> dict:
    embeddings = []
    token_lengths = []
    started = perf_counter()
    for batch_start in range(0, len(texts), batch_size):
        batch = texts[batch_start:batch_start + batch_size]
        if collect_lengths:
            lengths = dense_model["tokenizer"](
                batch, padding=False, truncation=False,
                add_special_tokens=True, return_length=True,
            )["length"]
            token_lengths.extend(int(length) for length in lengths)
        inputs = dense_model["tokenizer"](
            batch, padding=True, truncation=True, max_length=DENSE_MAX_LENGTH,
            return_tensors="pt",
        )
        inputs = {name: value.to(dense_model["device"]) for name, value in inputs.items()}
        with torch.no_grad():
            outputs = dense_model["model"](**inputs, return_dict=True)
            embedding = outputs.last_hidden_state[:, 0]
            embedding = F.normalize(embedding, p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    diagnostics = None
    if collect_lengths:
        lengths = np.asarray(token_lengths, dtype=np.int32)
        truncated_count = int(np.sum(lengths > DENSE_MAX_LENGTH))
        diagnostics = {
            "median": float(np.median(lengths)),
            "p95": float(np.percentile(lengths, 95)),
            "max": int(lengths.max()),
            "truncated_count": truncated_count,
            "truncated_fraction": truncated_count / len(lengths),
        }
    return {
        "embeddings": encoded, "seconds": perf_counter() - started,
        "token_lengths": diagnostics,
    }


def retrieve_dense(
    query_embeddings: torch.Tensor, corpus_embeddings: torch.Tensor,
    chunks: list[dict], sample_ids: list[str],
) -> dict:
    if query_embeddings.shape[0] != len(sample_ids):
        raise ValueError("query embedding count mismatch")
    started = perf_counter()
    passage_embedding = corpus_embeddings.to("cuda")
    candidates_by_query = {}
    for batch_start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[batch_start:batch_start + QUERY_BATCH_SIZE]
        query_embedding = query_embeddings[
            batch_start:batch_start + len(batch_ids)
        ].to("cuda")
        score = query_embedding @ passage_embedding.T
        if not torch.isfinite(score).all():
            raise RuntimeError("dense similarity produced non-finite scores")
        top_scores, top_indices = torch.topk(
            score, k=TOP_K_CHUNKS, dim=1, largest=True, sorted=True
        )
        for row, sample_id in enumerate(batch_ids):
            raw_hits = list(zip(
                top_scores[row].float().cpu().tolist(),
                top_indices[row].cpu().tolist(),
            ))
            raw_hits.sort(key=lambda item: (-item[0], item[1]))
            hits = []
            for rank, (score_value, index_value) in enumerate(raw_hits, start=1):
                chunk_index = int(index_value)
                hits.append({
                    "document_id": chunks[chunk_index]["document_id"],
                    "chunk_index": chunk_index,
                    "score": float(score_value),
                    "chunk_rank": rank,
                })
            candidates_by_query[sample_id] = aggregate_hits(hits)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    seconds = perf_counter() - started
    del passage_embedding
    torch.cuda.empty_cache()
    return package_candidates(candidates_by_query, seconds)


def rank_from_ce_scores(candidates: list[dict], score_lists: list[list[float]]) -> list[dict]:
    if len(candidates) != len(score_lists):
        raise RuntimeError("candidate/score count mismatch")
    ranked = []
    for candidate, scores in zip(candidates, score_lists):
        if len(scores) != len(candidate["supporting_chunk_indices"]):
            raise RuntimeError("supporting chunk/score count mismatch")
        if not 1 <= len(scores) <= SUPPORTING_CHUNKS_PER_DOCUMENT:
            raise RuntimeError("candidate must have one or two supporting chunks")
        if not all(isfinite(float(score)) for score in scores):
            raise RuntimeError("cross-encoder produced a non-finite score")
        ranked.append({
            "document_id": candidate["document_id"],
            "original_rank": candidate["original_rank"],
            "cross_encoder_score": sum(float(score) for score in scores),
        })
    ranked.sort(key=lambda item: (
        -item["cross_encoder_score"], item["original_rank"], item["document_id"]
    ))
    return ranked


def score_pairs(reranker: dict, pairs: list[tuple[str, str]]) -> dict:
    scores = []
    forward_seconds = 0.0
    started = perf_counter()
    for batch_start in range(0, len(pairs), RERANKER_BATCH_SIZE):
        batch = pairs[batch_start:batch_start + RERANKER_BATCH_SIZE]
        inputs = reranker["tokenizer"](
            [pair[0] for pair in batch], [pair[1] for pair in batch],
            padding=True, truncation="only_second",
            max_length=RERANKER_MAX_SEQUENCE_LENGTH, return_tensors="pt",
        )
        inputs = {name: value.to(reranker["device"]) for name, value in inputs.items()}
        torch.cuda.synchronize()
        forward_started = perf_counter()
        with torch.no_grad():
            logits = reranker["model"](**inputs, return_dict=True).logits.view(-1).float()
        torch.cuda.synchronize()
        forward_seconds += perf_counter() - forward_started
        values = logits.cpu().tolist()
        if len(values) != len(batch) or not all(isfinite(value) for value in values):
            raise RuntimeError("reranker returned invalid scores")
        scores.extend(float(value) for value in values)
    return {
        "scores": scores, "pairs": len(pairs),
        "forward_seconds": forward_seconds,
        "total_seconds": perf_counter() - started,
    }


def rerank_system(reranker: dict, samples: dict, candidates: dict, chunks: list[dict]) -> dict:
    pairs = []
    counts = []
    for sample_id, sample in samples.items():
        query_counts = []
        for candidate in candidates[sample_id]:
            indices = candidate["supporting_chunk_indices"]
            query_counts.append(len(indices))
            pairs.extend((sample["question"], chunks[index]["text"]) for index in indices)
        counts.append((sample_id, query_counts))
    scoring = score_pairs(reranker, pairs)
    offset = 0
    rankings = {}
    deterministic = True
    for sample_id, query_counts in counts:
        score_lists = []
        for count in query_counts:
            score_lists.append(scoring["scores"][offset:offset + count])
            offset += count
        ranked = rank_from_ce_scores(candidates[sample_id], score_lists)
        repeated = rank_from_ce_scores(candidates[sample_id], score_lists)
        rankings[sample_id] = [document["document_id"] for document in ranked]
        deterministic &= rankings[sample_id] == [d["document_id"] for d in repeated]
    if offset != len(scoring["scores"]):
        raise RuntimeError("not every cross-encoder score was consumed")
    return {"rankings": rankings, "scoring": scoring, "deterministic": deterministic}


def candidate_coverage(samples: dict, rankings: dict) -> dict:
    recalls = {depth: [] for depth in (10, 20, 50, 100)}
    for sample_id, sample in samples.items():
        ranked = rankings[sample_id]
        if len(ranked) != CANDIDATE_DEPTH or len(ranked) != len(set(ranked)):
            raise RuntimeError("candidate ranking must contain 100 unique IDs")
        gold = {str(document_id) for document_id in sample["answer"]}
        for depth, values in recalls.items():
            values.append(len(gold.intersection(ranked[:depth])) / len(gold))
    at_100 = np.asarray(recalls[100])
    return {
        **{f"recall_at_{depth}": float(np.mean(values)) for depth, values in recalls.items()},
        "zero_recall_rate_at_100": float(np.mean(at_100 == 0)),
        "full_recall_rate_at_100": float(np.mean(at_100 == 1)),
    }


def make_predictions(rankings: dict) -> dict:
    predictions = {}
    for sample_id, ranked in rankings.items():
        top_ids = [str(document_id) for document_id in ranked[:FINAL_K]]
        if len(top_ids) != FINAL_K or len(top_ids) != len(set(top_ids)):
            raise RuntimeError("top-5 prediction must contain five unique IDs")
        predictions[sample_id] = {"answer": top_ids}
    return predictions


def bundled_scorer_compatible_eval(predictions: dict, truth: dict) -> dict:
    y_pred = {key: value["answer"] for key, value in predictions.items()}
    y_true = {key: value for key, value in truth.items()}
    ids_preds = [key for key in y_pred]
    ids_truth = [key for key in y_true]
    if len(ids_preds) != len(ids_truth):
        raise RuntimeError("Samples in predictions do not match the reference")
    recall = np.array([
        len(set(y_true[key]) & set(y_pred.get(key, set()))) / len(y_true[key])
        if 0 < len(y_pred.get(key)) <= 5 else 0 for key in ids_truth
    ]).mean()
    precision = np.array([
        len(set(y_true[key]) & set(y_pred.get(key, set()))) / len(y_pred[key])
        if 0 < len(y_pred.get(key)) <= 5 else 0 for key in ids_preds
    ]).mean()
    return {"precision": float(precision), "recall": float(recall)}


def internal_metrics(samples: dict, rankings: dict) -> dict:
    recalls = {depth: [] for depth in (5, 10, 20, 50, 100)}
    reciprocal_ranks = []
    for sample_id, sample in samples.items():
        ranked = rankings[sample_id]
        if len(ranked) != CANDIDATE_DEPTH or len(ranked) != len(set(ranked)):
            raise RuntimeError("final ranking must contain 100 unique IDs")
        gold = {str(document_id) for document_id in sample["answer"]}
        for depth, values in recalls.items():
            values.append(len(gold.intersection(ranked[:depth])) / len(gold))
        first = next((rank for rank, doc_id in enumerate(ranked, 1) if doc_id in gold), None)
        reciprocal_ranks.append(0.0 if first is None else 1.0 / first)
    return {
        **{f"recall_at_{depth}": float(np.mean(values)) for depth, values in recalls.items()},
        "mrr": float(np.mean(reciprocal_ranks)),
        "mrr_scope": "fixed top-100; absent gold gives reciprocal rank 0",
    }


def percentile(values: list[int], q: int) -> float | None:
    return None if not values else float(np.percentile(np.asarray(values), q))


def first_gold_summary(samples: dict, rankings: dict) -> dict:
    found = []
    bins = Counter()
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        first = next(
            (rank for rank, doc_id in enumerate(rankings[sample_id], 1) if doc_id in gold), None
        )
        if first is None:
            bins["not_found"] += 1
        else:
            found.append(first)
            label = ("rank_1" if first == 1 else "rank_2_5" if first <= 5
                     else "rank_6_10" if first <= 10 else "rank_11_20" if first <= 20
                     else "rank_21_50" if first <= 50 else "rank_51_100")
            bins[label] += 1
    labels = ("rank_1", "rank_2_5", "rank_6_10", "rank_11_20",
              "rank_21_50", "rank_51_100", "not_found")
    return {
        "when_found": {
            "median": float(median(found)) if found else None,
            "p90": percentile(found, 90), "p95": percentile(found, 95),
        },
        "counts": {label: bins[label] for label in labels},
    }


def overlap_summary(left: dict, right: dict) -> dict:
    values = np.asarray([
        len(set(left[sample_id]) & set(right[sample_id])) for sample_id in left
    ])
    return {
        "mean": float(values.mean()), "median": float(np.median(values)),
        "p10": float(np.percentile(values, 10)),
        "p90": float(np.percentile(values, 90)),
    }


def candidate_records_finite(candidates: dict) -> bool:
    return all(
        isfinite(document["retrieval_score"])
        and all(isfinite(score) for score in document["supporting_chunk_scores"])
        for documents in candidates.values() for document in documents
    )


def close_to(value: float, expected: float) -> bool:
    return abs(value - expected) <= SANITY_ABS_TOLERANCE


def conclusion_for(precision_delta: float, recall_delta: float) -> str:
    tolerance = SANITY_ABS_TOLERANCE
    if precision_delta > tolerance and recall_delta > tolerance:
        return ("The DEV-selected Dense→CE configuration generalizes directionally "
                "on the fixed local holdout.")
    if precision_delta < -tolerance and recall_delta < -tolerance:
        return ("The DEV Dense→CE improvement does not generalize directionally "
                "on the fixed local holdout.")
    return ("The fixed local holdout shows a precision/recall trade-off between "
            "Dense→CE and the validated BM25→CE reference.")


In [ ]:
# Run the frozen aggregate-only holdout validation only on offline Kaggle.
run_started = perf_counter()
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle CUDA accelerator")
torch.cuda.reset_peak_memory_stats()

controls_tuple = (
    CHUNK_SIZE, CHUNK_OVERLAP, BM25_METHOD, BM25_K1, BM25_B, TOP_K_CHUNKS,
    DOCUMENT_AGGREGATION, CANDIDATE_DEPTH, SUPPORTING_CHUNKS_PER_DOCUMENT,
    DENSE_MAX_LENGTH, RERANKER_MAX_SEQUENCE_LENGTH, RERANKER_BATCH_SIZE, FINAL_K,
)
expected_controls_tuple = (
    2_000, 200, "lucene", 1.5, 0.75, 2_000, "sum_top_2", 100, 2,
    8_192, 8_192, 128, 5,
)
if controls_tuple != expected_controls_tuple:
    raise RuntimeError("a controlled experiment setting changed; stop")

holdout_samples, split_info = load_fixed_holdout(LEGALIR_SOURCE_PATH)
documents = load_corpus(CORPUS_PATH)
chunks = chunk_corpus(documents)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_CHUNKS:
    raise RuntimeError(
        f"fixed corpus mismatch: got {len(documents)} documents / {len(chunks)} chunks"
    )

bm25_index = build_bm25(chunks)
dense_model = load_dense_model()
dense_model_load_seconds = dense_model["load_seconds"]
dense_model_metadata = dict(dense_model["metadata"])
corpus_encoding = encode_normalized_cls(
    dense_model, [chunk["text"] for chunk in chunks],
    batch_size=CORPUS_BATCH_SIZE, collect_lengths=True,
)
corpus_embeddings = corpus_encoding["embeddings"]
if not torch.allclose(
    torch.linalg.vector_norm(corpus_embeddings.float(), dim=1),
    torch.ones(len(corpus_embeddings)), atol=2e-3, rtol=0,
):
    raise RuntimeError("corpus embeddings are not L2-normalized")

# Two-query implementation smoke test precedes the complete holdout query run.
smoke_samples = dict(list(holdout_samples.items())[:SMOKE_QUERIES])
smoke_bm25_first = retrieve_bm25(bm25_index, chunks, smoke_samples)
smoke_bm25_second = retrieve_bm25(bm25_index, chunks, smoke_samples)
smoke_query_encoding = encode_normalized_cls(
    dense_model, [sample["question"] for sample in smoke_samples.values()],
    batch_size=QUERY_BATCH_SIZE, collect_lengths=False,
)
smoke_dense_first = retrieve_dense(
    smoke_query_encoding["embeddings"], corpus_embeddings, chunks, list(smoke_samples)
)
smoke_dense_second = retrieve_dense(
    smoke_query_encoding["embeddings"], corpus_embeddings, chunks, list(smoke_samples)
)
move_model(dense_model, "cpu")
reranker = load_reranker()
smoke_lexical_ce = rerank_system(
    reranker, smoke_samples, smoke_bm25_first["candidates"], chunks
)
smoke_dense_ce = rerank_system(
    reranker, smoke_samples, smoke_dense_first["candidates"], chunks
)
smoke_norms = torch.linalg.vector_norm(
    smoke_query_encoding["embeddings"].float(), dim=1
)
smoke_test = {
    "queries": SMOKE_QUERIES,
    "finite_dense_embeddings": bool(
        torch.isfinite(corpus_embeddings).all()
        and torch.isfinite(smoke_query_encoding["embeddings"]).all()
    ),
    "normalized_dense_embeddings": bool(torch.allclose(
        smoke_norms, torch.ones(SMOKE_QUERIES), atol=2e-3, rtol=0
    )),
    "finite_retrieval_scores": bool(
        candidate_records_finite(smoke_bm25_first["candidates"])
        and candidate_records_finite(smoke_dense_first["candidates"])
    ),
    "lexical_100_unique_candidate_documents": all(
        len(ranking) == 100 and len(ranking) == len(set(ranking))
        for ranking in smoke_bm25_first["rankings"].values()
    ),
    "dense_100_unique_candidate_documents": all(
        len(ranking) == 100 and len(ranking) == len(set(ranking))
        for ranking in smoke_dense_first["rankings"].values()
    ),
    "finite_cross_encoder_scores": bool(
        all(isfinite(score) for score in smoke_lexical_ce["scoring"]["scores"])
        and all(isfinite(score) for score in smoke_dense_ce["scoring"]["scores"])
    ),
    "deterministic_ordering": bool(
        smoke_bm25_first["rankings"] == smoke_bm25_second["rankings"]
        and smoke_dense_first["rankings"] == smoke_dense_second["rankings"]
        and smoke_lexical_ce["deterministic"] and smoke_dense_ce["deterministic"]
    ),
    "no_duplicate_document_ids": all(
        len(ranking) == len(set(ranking))
        for result_part in (smoke_lexical_ce, smoke_dense_ce)
        for ranking in result_part["rankings"].values()
    ),
}
if not all(value for key, value in smoke_test.items() if key != "queries"):
    raise RuntimeError(f"smoke-test invariant failed: {smoke_test}")
move_model(reranker, "cpu")
move_model(dense_model, "cuda")

lexical_candidates = retrieve_bm25(bm25_index, chunks, holdout_samples)
query_encoding = encode_normalized_cls(
    dense_model, [sample["question"] for sample in holdout_samples.values()],
    batch_size=QUERY_BATCH_SIZE, collect_lengths=False,
)
dense_candidates = retrieve_dense(
    query_encoding["embeddings"], corpus_embeddings, chunks, list(holdout_samples)
)
move_model(dense_model, "cpu")
del dense_model
gc.collect()
torch.cuda.empty_cache()

lexical_coverage = candidate_coverage(holdout_samples, lexical_candidates["rankings"])
dense_coverage = candidate_coverage(holdout_samples, dense_candidates["rankings"])
if not close_to(
    lexical_coverage["recall_at_100"], EXPECTED_BM25_HOLDOUT_RECALL_AT_100
):
    raise RuntimeError(
        "historical BM25 holdout candidate Recall@100 sanity mismatch: "
        f"expected {EXPECTED_BM25_HOLDOUT_RECALL_AT_100}, "
        f"got {lexical_coverage['recall_at_100']}; stop"
    )
candidate_overlap = overlap_summary(
    lexical_candidates["rankings"], dense_candidates["rankings"]
)

move_model(reranker, "cuda")
lexical_ce = rerank_system(
    reranker, holdout_samples, lexical_candidates["candidates"], chunks
)
if any(
    set(lexical_candidates["rankings"][sample_id])
    != set(lexical_ce["rankings"][sample_id])
    for sample_id in holdout_samples
):
    raise RuntimeError("lexical cross-encoder changed its candidate set")
holdout_truth = {
    sample_id: sample["answer"] for sample_id, sample in holdout_samples.items()
}
lexical_bundled = bundled_scorer_compatible_eval(
    make_predictions(lexical_ce["rankings"]), holdout_truth
)
lexical_internal = internal_metrics(holdout_samples, lexical_ce["rankings"])
if lexical_coverage["recall_at_100"] != lexical_internal["recall_at_100"]:
    raise RuntimeError("lexical Recall@100 changed after cross-encoder; implementation bug")
historical_bm25_checks = (
    (lexical_bundled["precision"], EXPECTED_BM25_HOLDOUT_CE_PRECISION),
    (lexical_bundled["recall"], EXPECTED_BM25_HOLDOUT_CE_RECALL),
    (lexical_internal["mrr"], EXPECTED_BM25_HOLDOUT_CE_MRR),
)
if not all(close_to(actual, expected) for actual, expected in historical_bm25_checks):
    actual = {
        "precision": lexical_bundled["precision"],
        "recall": lexical_bundled["recall"],
        "mrr": lexical_internal["mrr"],
    }
    expected = {
        "precision": EXPECTED_BM25_HOLDOUT_CE_PRECISION,
        "recall": EXPECTED_BM25_HOLDOUT_CE_RECALL,
        "mrr": EXPECTED_BM25_HOLDOUT_CE_MRR,
    }
    raise RuntimeError(
        f"historical BM25→CE fixed-holdout sanity mismatch: "
        f"expected {expected}, got {actual}; stop before Dense comparison"
    )

dense_ce = rerank_system(
    reranker, holdout_samples, dense_candidates["candidates"], chunks
)
if any(
    set(dense_candidates["rankings"][sample_id])
    != set(dense_ce["rankings"][sample_id])
    for sample_id in holdout_samples
):
    raise RuntimeError("dense cross-encoder changed its candidate set")
dense_bundled = bundled_scorer_compatible_eval(
    make_predictions(dense_ce["rankings"]), holdout_truth
)
dense_internal = internal_metrics(holdout_samples, dense_ce["rankings"])
if dense_coverage["recall_at_100"] != dense_internal["recall_at_100"]:
    raise RuntimeError("dense Recall@100 changed after cross-encoder; implementation bug")

deltas = {
    "precision": dense_bundled["precision"] - lexical_bundled["precision"],
    "recall": dense_bundled["recall"] - lexical_bundled["recall"],
    "mrr": dense_internal["mrr"] - lexical_internal["mrr"],
    **{
        f"recall_at_{depth}": (
            dense_internal[f"recall_at_{depth}"]
            - lexical_internal[f"recall_at_{depth}"]
        ) for depth in (10, 20, 50, 100)
    },
}
coverage_to_final = {
    "lexical_cross_encoder": {
        "candidate_ceiling_recall_at_100": lexical_coverage["recall_at_100"],
        "final_recall_at_5": lexical_internal["recall_at_5"],
        "remaining_ranking_gap": (
            lexical_coverage["recall_at_100"] - lexical_internal["recall_at_5"]
        ),
    },
    "dense_cross_encoder": {
        "candidate_ceiling_recall_at_100": dense_coverage["recall_at_100"],
        "final_recall_at_5": dense_internal["recall_at_5"],
        "remaining_ranking_gap": (
            dense_coverage["recall_at_100"] - dense_internal["recall_at_5"]
        ),
    },
}

result = {
    "split": split_info,
    "controls": {
        "documents": len(documents), "chunks": len(chunks),
        "chunk_size_characters": CHUNK_SIZE, "overlap_characters": CHUNK_OVERLAP,
        "source_preserving_windows": True,
        "bm25": {
            "library": f"bm25s=={bm25s.__version__}", "method": BM25_METHOD,
            "k1": BM25_K1, "b": BM25_B,
            "tokenization": r"lowercase Unicode \w+",
        },
        "retrieval_top_k_chunks": {"lexical": TOP_K_CHUNKS, "dense": TOP_K_CHUNKS},
        "document_aggregation": "sum of top two retriever chunk scores",
        "candidate_documents": CANDIDATE_DEPTH,
        "supporting_chunks_per_document": "up to 2 from each retriever's own hit pool",
        "cross_encoder_aggregation": "sum of up to two independent chunk scores",
        "cross_encoder_tie_break": [
            "CE score descending", "original retriever rank ascending",
            "document_id ascending",
        ],
        "final_k": FINAL_K,
        "bundled_scorer_semantics": {
            "recall_q": "|set(gold) intersect set(pred)| / len(gold)",
            "precision_q": "|set(gold) intersect set(pred)| / len(pred)",
            "aggregation": "macro mean",
        },
        "aggregate_only_holdout": True,
        "no_retrieval_score_fusion": True,
        "no_method_selection_or_tuning": True,
    },
    "models": {
        "dense": {
            **dense_model_metadata,
            "representation": "L2-normalized CLS hidden state",
            "similarity": "query_embedding @ passage_embedding.T",
            "query_instruction": None, "max_length": DENSE_MAX_LENGTH,
            "dynamic_padding": True, "dtype": "float16", "device": "cuda",
        },
        "reranker": {
            **reranker["metadata"],
            "max_sequence_length": RERANKER_MAX_SEQUENCE_LENGTH,
            "dtype": "float16", "batch_size": RERANKER_BATCH_SIZE,
            "device": "cuda",
        },
    },
    "smoke_test": smoke_test,
    "candidate_retrieval": {
        "lexical": lexical_coverage, "dense": dense_coverage,
        "overlap": candidate_overlap,
        "cross_encoder_preserves_recall_at_100": {
            "lexical": True, "dense": True,
        },
    },
    "lexical_cross_encoder": {
        "bundled_scorer": lexical_bundled, "internal": lexical_internal,
        "first_gold_rank": first_gold_summary(holdout_samples, lexical_ce["rankings"]),
    },
    "dense_cross_encoder": {
        "bundled_scorer": dense_bundled, "internal": dense_internal,
        "first_gold_rank": first_gold_summary(holdout_samples, dense_ce["rankings"]),
    },
    "deltas": deltas,
    "coverage_to_final": coverage_to_final,
    "runtime": {
        "bm25_build_seconds": bm25_index["build_seconds"],
        "bm25_retrieval_seconds": lexical_candidates["seconds"],
        "bm25_build_and_retrieval_seconds": (
            bm25_index["build_seconds"] + lexical_candidates["seconds"]
        ),
        "dense_model_load_seconds": dense_model_load_seconds,
        "dense_corpus_encoding_seconds": corpus_encoding["seconds"],
        "dense_query_encoding_seconds": query_encoding["seconds"],
        "dense_retrieval_seconds": dense_candidates["seconds"],
        "reranker_model_load_seconds": reranker["load_seconds"],
        "lexical_cross_encoder_pairs": lexical_ce["scoring"]["pairs"],
        "lexical_cross_encoder_forward_seconds": lexical_ce["scoring"]["forward_seconds"],
        "dense_cross_encoder_pairs": dense_ce["scoring"]["pairs"],
        "dense_cross_encoder_forward_seconds": dense_ce["scoring"]["forward_seconds"],
        "dense_embedding_dimension": int(corpus_embeddings.shape[1]),
        "dense_embedding_dtype": str(corpus_embeddings.dtype).removeprefix("torch."),
        "dense_corpus_token_length_before_truncation": corpus_encoding["token_lengths"],
        "total_runtime_seconds": perf_counter() - run_started,
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "corpus_batch_size": CORPUS_BATCH_SIZE, "query_batch_size": QUERY_BATCH_SIZE,
        "torch_version": torch.__version__, "transformers_version": transformers.__version__,
    },
    "conclusion": conclusion_for(
        deltas["precision"], deltas["recall"]
    ),
}
RESULT_PATH.write_text(
    json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(result, ensure_ascii=False, indent=2))
print("Saved:", RESULT_PATH)
